In [1]:
## Setup — resolve project paths"

import sys
from pathlib import Path

def find_src_dir(start: Path = None) -> Path:
    """Walk upward from the current directory until a 'src' folder is found."""
    current = start or Path.cwd()
    for _ in range(5):
        candidate = current / "src"
        if candidate.exists():
            return candidate
        current = current.parent
    raise FileNotFoundError("Could not locate a 'src' folder above the current directory.")

SRC_DIR = find_src_dir()
sys.path.append(str(SRC_DIR))

import polars as pl
from paths import PROJECT_ROOT, ZIP_DIR, DATA_DIR, EXTRACTED_DIR, PARQUET_DIR, IBES_DIR, CRSP_DIR

print(f"Project root: {PROJECT_ROOT}")

Project root: C:\Users\axels\CBOE_DATA_ANALYSIS_ASPARN_FINAL


In [2]:
## Step 1: Rebuild CBOE pipeline (only needed for first time setup)

from pipeline.extract_zips import run_extraction
from pipeline.ingest_cboe import run_ingestion

if not any(PARQUET_DIR.glob("*.parquet")):
    run_extraction()
    run_ingestion(raw_dir=str(EXTRACTED_DIR), out_dir=str(PARQUET_DIR))
else:
    print("Parquet files already exist -- skipping. Delete data/cboe_parquet/ first if you want to rebuild.")

In [3]:
## Step 2: Verify CBOE data is intact

from pipeline.verify_setup import main
main()

In [ ]:
## Step 3: Ingest IBES analyst forecasts

from pipeline.ingest_ibes import run_ibes_ingestion

if not (IBES_DIR / "ibes_clean.parquet").exists():
    run_ibes_ingestion()
else:
    print("ibes_clean.parquet already exists -- skipping.")

In [ ]:
## Step 4: Build the firm event dispersion panel 

from pipeline.build_dispersion_events import build_dispersion_events

if not (IBES_DIR / "dispersion_events.parquet").exists():
    build_dispersion_events()
else:
    print("dispersion_events.parquet already exists -- skipping.")

In [ ]:
## Step 5: Build the daily participant actiivty table (retail + procust)

from pipeline.build_daily_retail_activity import build_daily_retail_activity

daily_dir = DATA_DIR / "cboe_daily_retail"
if not list(daily_dir.glob("*.parquet")):
    build_daily_retail_activity()
else:
    print("daily_retail_*.parquet already exists -- skipping. "
          "Delete the folder first if you've changed build_daily_retail_activity.py.")

In [ ]:
## Step 6: Ingest CRSP daily prices

from pipeline.ingest_crsp import run_crsp_ingestion

if not (CRSP_DIR / "crsp_daily.parquet").exists():
    run_crsp_ingestion()
else:
    print("crsp_daily.parquet already exists -- skipping. "
          "Delete it first if you need to re-ingest.")

In [ ]:
## Step 7: Compute moneyness (joins CRSP spot prices to option level CBOE data)

from pipeline.build_moneyness import build_moneyness

mny_dir = DATA_DIR / "cboe_daily_moneyness"
if not list(mny_dir.glob("*.parquet")):
    build_moneyness()
else:
    print("daily_moneyness_*.parquet already exists -- skipping. "
          "Delete the folder first if you've changed build_moneyness.py.")

In [ ]:
## Step 8: Verify moneyness coverage

# The event sample percentage is what certifies DV2 is useable. Unpriced volume outside the event sample is index products ((^SPX, ^VIX etc.) and is expected).

from analysis.check_moneyness_coverage import check_coverage
top_unknown = check_coverage()